In [1]:
import matplotlib.pyplot as plt
import numpy as np
import cv2


In [ ]:
# Image paths
bad_restoration_image_path  = "/data/atran16/ProteinClassification_3D/3D_PDB_5013/testingDataFromProfessorSu/HYDROLASE/8dnm/6.jpg"
good_restoration_image_path = "/data/atran16/ProteinClassification_3D/3D_PDB_5013/testingDataFromProfessorSu/HYDROLASE/8dnm/7.png"

img_bad  = cv2.imread(bad_restoration_image_path)
img_good = cv2.imread(good_restoration_image_path)

img_bad_rgb  = cv2.cvtColor(img_bad,  cv2.COLOR_BGR2RGB)
img_good_rgb = cv2.cvtColor(img_good, cv2.COLOR_BGR2RGB)

########################################
# Remove background using GrabCut
########################################
mask     = np.zeros(img_bad.shape[:2], np.uint8)
bgdModel = np.zeros((1, 65), np.float64)
fgdModel = np.zeros((1, 65), np.float64)

h, w = img_bad.shape[:2]
rect = (10, 10, w-20, h-20)
cv2.grabCut(img_bad, mask, rect, bgdModel, fgdModel, 10, cv2.GC_INIT_WITH_RECT)

# 0/2 = background, 1/3 = foreground
mask2 = np.where((mask == 2) | (mask == 0), 0, 1).astype('uint8')

# Morphological cleanup
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
mask2  = cv2.morphologyEx(mask2, cv2.MORPH_CLOSE, kernel, iterations=3)
mask2  = cv2.morphologyEx(mask2, cv2.MORPH_OPEN,  kernel, iterations=1)

# Apply mask (background → black)
img_bad_nobg     = img_bad * mask2[:, :, np.newaxis]
img_bad_nobg_rgb = cv2.cvtColor(img_bad_nobg, cv2.COLOR_BGR2RGB)

########################################
# Show results  (7.png is reference only — not used in processing)
########################################
plt.figure(figsize=(12, 5))

plt.subplot(1, 3, 1)
plt.imshow(img_bad_rgb)
plt.title("Original Bad Image")
plt.axis('off')

plt.subplot(1, 3, 2)
plt.imshow(img_bad_nobg_rgb)
plt.title("Bad Image (Background Removed)")
plt.axis('off')

plt.subplot(1, 3, 3)
plt.imshow(img_good_rgb)
plt.title("Good Image (7.png) — reference only")
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import os

out_path = '/data/atran16/ProteinClassification_3D/visuallization/testVisuallizaion/background_removed.png'
cv2.imwrite(out_path, img_bad_nobg)
print(f'Saved: {out_path}')